## Re-ranking for better RAG response
### Background / Goal
Retrieval Augmented Generation (RAG) allows the user to query unstructured data and return a clear response.</br></br>
Providing a worthwhile response is a greater challenge when the underlying data is not only unstructured, but has a very high noise component from memes, rage posts, and poorly organized thoughts as the LLM needs to filter for the nuggets of substance.</br></br>
Our goal is to show the benefit of performing reranking to generate a better response to a user question. Thus we want our experiment to:
<ul>
<li>Generate a response based **solely** on the retrieved statements</li>
<li>Use a particularly noisy dataset so as to make effective retrieval challenging</li>
</ul>
To this end, we do the following:
<ul>
<li>Build the RAG systems to use the same documents, embeddings, and vector store</li>
<li>Perform initial retrieval with top-K = 'WIDE_K'; the larger group of documents to use in reranking</li>
<li>For the 'vanilla RAG', select only the first 'FINAL_K' documents.</li>
<li>For the 'reranked RAG', perform ranking on the document list and downselect to 'FINAL_K' based on score.</li>
<li>Compare quality of result from both paths.</li>
</ul>

### Import & Globals

In [1]:
import dotenv
import json
import openai
import os
from pathlib import Path
import re
import requests
import torch

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel
)
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama
from langchain_openai import OpenAIEmbeddings


# Globals
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
device = "mps" if torch.backends.mps.is_available() else "cpu"
REDDIT_DIR = Path("data/reddit_manual")
WIDE_K = 25  # top-k to retrieve before reranking
FINAL_K = 5 # top-k to return after reranking (or if no reranking)

SAVE_THREADS = False  # whether to save Reddit threads to markdown files

### Helpers

In [2]:

def to_json_url(url: str) -> str:
    '''Convert a Reddit thread URL to its JSON endpoint.'''
    url = url.strip()
    if url.endswith("/"):
        url = url[:-1]
    if not url.endswith(".json"):
        url = url + ".json"
    # helpful sometimes:
    if "?" not in url:
        url = url + "?raw_json=1"
    return url

def clean(text: str) -> str:
    '''Clean Reddit comment text by removing extra whitespace.'''
    text = re.sub(r"\s+", " ", text or "").strip()
    return text

def extract_comments(node, acc, min_score=1, depth=0, only_top_level=False):
    '''Extract comments from Reddit JSON structure.'''
    kind = node.get("kind")
    data = node.get("data", {})

    if kind == "t1":  # comment
        score = data.get("score", 0)
        body = clean(data.get("body", ""))

        if body and score >= min_score:
            if (not only_top_level) or (depth == 0):
                acc.append(body)

        replies = data.get("replies")
        if isinstance(replies, dict):
            children = replies.get("data", {}).get("children", [])
            for child in children:
                extract_comments(
                    child,
                    acc,
                    min_score=min_score,
                    depth=depth + 1,
                    only_top_level=only_top_level,
                )

    elif kind in {"Listing"}:
        children = data.get("children", [])
        for child in children:
            extract_comments(
                child,
                acc,
                min_score=min_score,
                depth=depth,
                only_top_level=only_top_level,
            )

def extract_comments_with_parent(node, acc, min_score=1, parent_text="", depth=0):
    '''Extract comments from Reddit JSON structure, including parent comment snippet.'''
    kind = node.get("kind")
    data = node.get("data", {})

    if kind == "t1":
        score = data.get("score", 0)
        body = clean(data.get("body", ""))

        if body and score >= min_score:
            parent_snip = parent_text[:220]
            acc.append({
                "depth": depth,
                "score": score,
                "parent": parent_snip,
                "body": body,
            })

        replies = data.get("replies")
        if isinstance(replies, dict):
            children = replies.get("data", {}).get("children", [])
            for child in children:
                extract_comments_with_parent(
                    child, acc,
                    min_score=min_score,
                    parent_text=body,
                    depth=depth + 1
                )

    elif kind == "Listing":
        for child in data.get("children", []):
            extract_comments_with_parent(child, acc, min_score=min_score)



def download_thread(url: str, min_score=1, only_top_level=False, include_parent=False):
    '''Download Reddit thread comments from JSON endpoint.'''
    json_url = to_json_url(url)
    headers = {"User-Agent": "eli5-reranker-demo/0.1 (personal research)"}
    resp = requests.get(json_url, headers=headers, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    post_listing = payload[0]
    comments_listing = payload[1]

    post_children = post_listing.get("data", {}).get("children", [])
    title = "Reddit Thread"
    if post_children:
        title = post_children[0].get("data", {}).get("title", title)

    comments = []
    for child in comments_listing.get("data", {}).get("children", []):
        if include_parent:
            extract_comments_with_parent(
                child, comments, min_score=min_score, parent_text="", depth=0
            )
        else:
            extract_comments(
                child, comments,
                min_score=min_score, depth=0, only_top_level=only_top_level
            )
    return title, comments


def write_markdown(title, comments, out_path):
    '''Write Reddit thread title and comments to a Markdown file.'''
    lines = []
    lines.append(f"# Thread\nTitle: {title}\n")
    lines.append("## Comments")

    if not comments:
        out_path.write_text("\n".join(lines), encoding="utf-8")
        return

    # Parent-aware dict format
    if isinstance(comments[0], dict):
        for c in comments:
            body = c.get("body", "")
            parent = c.get("parent", "")
            depth = c.get("depth", 0)
            score = c.get("score", 0)

            if depth == 0:
                lines.append(f"- ({score}) {body}")
            else:
                lines.append(f"- ({score}) Reply to: {parent} | {body}")

    # Simple string format
    else:
        for c in comments:
            lines.append(f"- {c}")

    out_path.write_text("\n".join(lines), encoding="utf-8")



def save_thread(url, min_score=1, only_top_level=False, include_parent=False) -> None:
    '''Download Reddit thread and save comments to a Markdown file.'''
    title, comments = download_thread(url, min_score=min_score, only_top_level=only_top_level, include_parent=include_parent)

    safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", title).strip("_")
    out_path = REDDIT_DIR / f"{safe_name[:80]}.md"

    write_markdown(title, comments, out_path)

    print(f"Saved {len(comments)} comments to: {out_path}")


### Download threads from Reddit

In [3]:
if SAVE_THREADS:
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1gjkn0t/eli5_how_do_tariffs_affect_the_price_of_goods/.json", 
                min_score=3, only_top_level=False, include_parent=True)
    save_thread("https://www.reddit.com/r/AskUS/comments/1jra630/eli5_what_is_a_tariff_and_why_it_is_a_badgood/.json", 
                min_score=2, only_top_level=False, include_parent=True)
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1mtj95m/eli5_where_does_crypto_get_its_value/.json", 
                min_score=2, only_top_level=False, include_parent=True)
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1hag4h9/eli5where_is_all_the_money_for_crypto_coming_from/.json", 
                min_score=3, only_top_level=False, include_parent=True)
    save_thread("https://www.reddit.com/r/investing/comments/ukfxps/can_someone_explain_why_buffet_thinks_bitcoin/.json",
                min_score=4, only_top_level=False, include_parent=True)
    save_thread("https://www.reddit.com/r/stocks/comments/1if3j75/trumps_new_tariffs_how_are_you_adjusting_your/.json", 
                min_score=4, only_top_level=False, include_parent=True)

In [4]:
def is_bad_comment(body: str) -> bool:
    """
    Determine if a Reddit comment body is "bad" (removed, deleted, or too short).
    Returns True if the comment is bad, False otherwise.
    """
    b = (body or "").strip().lower()
    return b in {"[removed]", "[deleted]"} or len(b) < 8

def load_reddit_comment_docs(directory=REDDIT_DIR, include_post_snippet=True):
    docs = []
    for path in sorted(directory.glob("*.md")):
        text = path.read_text(encoding="utf-8")

        # Title
        title_match = re.search(r"^Title:\s*(.+)$", text, flags=re.MULTILINE)
        title = title_match.group(1).strip() if title_match else path.stem

        # Post block (optional)
        post_text = ""
        if include_post_snippet:
            post_match = re.search(
                r"^##\s+Post\s*$([\s\S]*?)(?=^##\s+Comments\s*$|$)",
                text,
                flags=re.MULTILINE,
            )
            if post_match:
                post_text = post_match.group(1).strip()

        # Comments section
        parts = re.split(r"^##\s+Comments\s*$", text, flags=re.MULTILINE)
        if len(parts) < 2:
            continue
        comments_blob = parts[1]

        bullets = re.findall(r"^\s*-\s+(.*\S.*)$", comments_blob, flags=re.MULTILINE)

        # Small, consistent context prefix
        post_snippet = ""
        if post_text:
            post_snippet = post_text[:280].replace("\n", " ")

        for i, b in enumerate(bullets):
            comment = b.strip()
            if not comment:
                continue
            if is_bad_comment(comment):
                continue
            context_prefix = f"Thread: {title}\n"
            if post_snippet:
                context_prefix += f"Post snippet: {post_snippet}\n"

            docs.append(
                Document(
                    page_content=context_prefix + f"Comment: {comment}",
                    metadata={
                        "source": path.name,
                        "thread_title": title,
                        "comment_idx": i,
                    },
                )
            )
    return docs

docs = load_reddit_comment_docs()
print("Loaded comment docs:", len(docs))


Loaded comment docs: 527


### Set up vector store and baseline retriever

In [5]:
dotenv.load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": WIDE_K})


#### LLM hook

In [6]:

# You’d need an Ollama model name you’ve pulled locally.
# Example name might differ in your setup.
llm = ChatOllama(model="mistral", temperature=0)


#### LLM Reranker

In [7]:
TOPIC_DEFS = {
    "tariffs": "A tariff is a tax on imported goods, typically paid by the importer.",
    "crypto_value": "Crypto value is market-driven and not backed by a claim on cash flows in the way many traditional assets are.",
    "fed_rates": (
        "The federal funds rate is the interest rate at which banks lend "
        "reserve balances to each other overnight. By changing this rate, "
        "the Fed influences other interest rates, borrowing costs, and "
        "overall demand in the economy."
    ),
    "gold_standard": "A gold standard links a country's currency to a fixed quantity of gold, limiting discretionary expansion of the money supply.",
}

SYSTEM_RANKER_PROMPT = """
You are a careful, neutral evaluator of short-form public comments about economics and finance.
Your job is to rank how useful a comment is for answering a question.

You may use concise definition anchors provided to you to detect comments that deny or muddle basic terminology.
These anchors are not the full answer. They are only used to score definitional correctness.

Prioritize:
- correctness of the core definition
- clear mechanism explained in steps
- simple examples
- balanced caveats/tradeoffs

Avoid rewarding:
- absolutist rhetoric
- partisan cheering
- confident but vague claims
- off-topic dunking

Follow the scoring rules and return only the required JSON.
Use the full range. Most comments should fall between 4 and 8.
""".strip()


rank_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RANKER_PROMPT),
    ("human", """
User question:
{question}

Thread title:
{thread_title}

Definition anchor:
{core_definition}

Comment:
{comment}

Return JSON exactly like:
{{"score": 7}}

Scoring rules:
- If the comment denies or muddies the definition anchor, score it 0–3.
- High scores (9–10) require:
  1) Correct definition,
  2) Clear mechanism in steps,
  3) A simple example or consequence,
  4) At least one caveat/tradeoff.
- Avoid rewarding absolutist rhetoric.
Use the full range. Most comments should fall between 4 and 8.

Return ONLY the JSON object.
""".strip())
])


def score_one(question, doc, core_definition):
    msg = rank_prompt.invoke({
        "question": question,
        "thread_title": doc.metadata.get("thread_title", ""),
        "core_definition": core_definition,
        "comment": doc.page_content,
    })
    out = llm.invoke(msg).content

    try:
        score = int(json.loads(out).get("score", 0))
    except Exception:
        score = 0

    return max(0, min(10, score))


def score(question, candidates, core_definition):
    '''Score documents based on relevance to the question and core definition.'''
    scored = []
    for d in candidates:
        s = score_one(question, d, core_definition)
        scored.append((s, d))
    return scored




#### Create screenshot ready output

In [8]:
def show_ranked(label, items, limit=None):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    if items and isinstance(items[0], tuple):
        # (score, doc)
        rows = items[:limit] if limit else items
        for i, (score, d) in enumerate(rows, start=1):
            text = d.page_content.replace("\n", " ")
            text = (text[:180] + "…") if len(text) > 180 else text
            print(f"{i:02d}. [{score:>2}/10] {d.metadata['source']} | {text}")
    else:
        # docs
        rows = items[:limit] if limit else items
        for i, d in enumerate(rows, start=1):
            text = d.page_content.replace("\n", " ")
            text = (text[:180] + "…") if len(text) > 180 else text
            print(f"{i:02d}. {d.metadata['source']} | {text}")

def answer_from_docs(question, docs):
    context = "\n".join([d.page_content for d in docs])
    prompt = f"""Use only the context below to answer the question.

Context:
{context}

Question:
{question}

Give a concise answer in 3–5 sentences.
"""
    return llm.invoke(prompt).content


#### The LCEL “compare retrieval” mini-chain

In [23]:


def retrieve_candidates(question):
    return retriever.invoke(question)

compare_chain = RunnableParallel(
    question=RunnablePassthrough(),
    candidates=RunnableLambda(retrieve_candidates),
)

def answer_from_docs(question, docs):
    context = "\n".join([d.page_content for d in docs])
    prompt = f"""
You are summarizing Reddit comments.

Context (Reddit comments):
{context}

Step 1: List bullet points of key ideas that appear in the comments.
For each bullet, include a SHORT quote (3–7 words) from the comments showing
where the idea comes from.

Step 2: Using ONLY those bullet points, write a 3–4 sentence summary (~80-100 words).
Do NOT introduce any new ideas or terms that are not present in Step 1.
If an idea is not supported by a quote in Step 1, you may not use it.

Do NOT invent arguments that are not clearly present in the comments.


Begin with Step 1.

"""
    prompt_ = f"""
You are summarizing Reddit comments.

Context (Reddit comments):
{context}

    Step 1: Summarize the main portfolio moves commenters suggest and their reasons
(2–3 sentences).

Step 2: Based ONLY on those ideas, recommend a single strategy for a diversified
long-term investor. Explain why it best reflects the strongest arguments in the
comments, and mention the key tradeoffs (2–3 sentences).

Do NOT introduce new strategies or macro claims that do not appear in the comments.

Begin with Step 1.
"""
    return llm.invoke(prompt).content



def run_compare(question, topic):
    core_definition = TOPIC_DEFS[topic]

    # 1) Retrieve a wide set once
    docs = retriever.invoke(question)          # length ≈ WIDE_K
    # Assign scores to docs
    scored = score(question, docs, core_definition)   # list of (score, doc)

    # 2) Vanilla = first FINAL_K embedding hits
    vanilla_docs = scored[:FINAL_K]
    show_ranked(f"Vanilla (top-{FINAL_K} by embedding)", vanilla_docs, limit=FINAL_K)

    # 3) Rerank the wide pool, then take top FINAL_K
    scored.sort(key=lambda x: x[0], reverse=True)

    reranked_top = scored[:FINAL_K]
    show_ranked(f"Reranked (top-{FINAL_K} from k={WIDE_K})", reranked_top, limit=FINAL_K)

    vanilla_answer = answer_from_docs(question, [d for _, d in vanilla_docs])
    rerank_answer = answer_from_docs(question, [d for _, d in reranked_top])

    print("\n--- Vanilla answer ---\n", vanilla_answer)
    print("\n--- Reranked answer ---\n", rerank_answer)

    return vanilla_docs, reranked_top





### Run

In [14]:
# topic types = ["tariffs", "crypto_value", "fed_rates", "gold_standard"]
q = "How do tariffs impact consumers, producers, and the overall economy? Explain the mechanisms and tradeoffs."
run_compare(q, "tariffs")




Vanilla (top-5 by embedding)
01. [ 7/10] ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: (3) Reply to: [deleted] | There is denying it, because that doesn’t happen. They might temporarily benefit **some s…
02. [ 7/10] ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: (11) A tariff is usually used to encourage the sale of locally produced goods. If imported widgets are cheaper than…
03. [ 7/10] ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: (20) Reply to: assuming that there even are any domestic t-shirt manufacturers to compete. (yes, there are, but for…
04. [ 8/10] ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: (5) Reply to: Tariff is an tax imposed on goods at customs. Say your country imposes 10% tariff on imported

([(7,
   Document(id='43603760-74c8-49db-9a37-f009bf1bc615', metadata={'source': 'ELI5_How_do_tariffs_affect_the_price_of_goods.md', 'thread_title': 'ELI5: How do tariffs affect the price of goods?', 'comment_idx': 12}, page_content='Thread: ELI5: How do tariffs affect the price of goods?\nComment: (3) Reply to: [deleted] | There is denying it, because that doesn’t happen. They might temporarily benefit **some specific agent** but the total benefit is less than the total economic benefit of a market clearing equilibrium price. It might provide more jobs in the Widget building industry, but it will cost more jobs in the Dwaddle industry. Or it will cost consumers more, or consumers will buy less or a combination of those two. It might provide a temporary boost to government revenues, but the economic loss as a whole (to all agents in the system) will be greater than the gain to one agent. Tariffs impose a deadweight loss on the economy by setting the market price above the equilibrium l

In [24]:
q = "Does crypto provide better risk-adjusted returns than equities? Explain simply."
run_compare(q, "crypto_value")



Vanilla (top-5 by embedding)
01. [ 5/10] ELI5_Where_does_crypto_get_its_value.md | Thread: ELI5: Where does crypto get its value? Comment: (5) Reply to: It relies on an influx of more people in order to keep the momentum. They all have similarities and are not sa…
02. [ 7/10] ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: (234) Reply to: It's coming from people putting money into crypto. The problem is you are assuming that investi…
03. [ 5/10] ELI5_Where_does_crypto_get_its_value.md | Thread: ELI5: Where does crypto get its value? Comment: (3) Reply to: Sure, but you're still consistently using the wrong words. ;p | That’s because there isn’t really a term that …
04. [ 7/10] ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: (25) Reply to: Crypto is a zero-sum game. It's speculation and not actual "investing". All money coming out of 

([(5,
   Document(id='2f50f781-c86f-4a2d-ad54-25024177b261', metadata={'source': 'ELI5_Where_does_crypto_get_its_value.md', 'thread_title': 'ELI5: Where does crypto get its value?', 'comment_idx': 22}, page_content="Thread: ELI5: Where does crypto get its value?\nComment: (5) Reply to: It relies on an influx of more people in order to keep the momentum. They all have similarities and are not sane investments regardless of if you can make a fortune participating. | Sure, but you're still consistently using the wrong words. ;p")),
  (7,
   Document(id='4cf0a96c-cfd4-4086-bb61-bac7a58c6e41', metadata={'source': 'ELI5_Where_is_all_the_money_for_Crypto_coming_from.md', 'thread_title': 'ELI5:Where is all the money for Crypto coming from?', 'comment_idx': 1}, page_content="Thread: ELI5:Where is all the money for Crypto coming from?\nComment: (234) Reply to: It's coming from people putting money into crypto. The problem is you are assuming that investing is a zero-sum game. That money going in

In [ ]:
#q = "When should the Fed raise or lower interest rates, and what’s the mechanism by which this affects the economy?"
#run_compare(q, "fed_rates")
q = "According to these comments, what portfolio changes do they suggest an investor make in response to Trump’s new tariffs, and what is the single best strategy you would recommend based on their reasoning?"
run_compare(q, "tariffs")
                            


Vanilla (top-5 by embedding)
01. [ 7/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (9) Reply to: The funniest part about this is that Trump wants rate cuts. But Fed can’t do that when …
02. [ 5/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (12) Reply to: Lived through the dotcom & housing busts. I feel like tech is artificially propped up …
03. [ 7/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (8) Problem is that there is no safe haven. Interest rates will push up because tariffs = inflation =…
04. [ 6/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (6) Reply to: Honestly scrolled far too l

([(7,
   Document(id='c14b2f0e-074b-4033-9918-b51305e9592d', metadata={'source': 'Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md', 'thread_title': 'Trump’s New Tariffs – How Are You Adjusting Your Investments?', 'comment_idx': 119}, page_content='Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments?\nComment: (9) Reply to: The funniest part about this is that Trump wants rate cuts. But Fed can’t do that when inflation rises. And guess who caused it? Orange Baldy Personally I will stay invested into the S&P500 though. Storms come and go. St | The Fed *shouldn’t* do that when inflation rises. But now? Who the fuck knows anymore?')),
  (5,
   Document(id='dea38162-df48-46f2-849b-9c24a5f5084a', metadata={'source': 'Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md', 'thread_title': 'Trump’s New Tariffs – How Are You Adjusting Your Investments?', 'comment_idx': 47}, page_content="Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments?

In [17]:
q = "Based on these comments, how should a diversified long-term investor adjust their portfolio in response to Trump’s new tariffs, and why?"
run_compare(q, "tariffs")


Vanilla (top-5 by embedding)
01. [ 7/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (5) NVDA has almost propped up the market in its own. It was bound for a correction, as is the rest o…
02. [ 1/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (133) Swing trading mode for next 3 years
03. [ 2/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (5) Reply to: [deleted] | What sectors are stockpiling right now?
04. [ 7/10] Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: (81) As a geezer \[born in 1949, I started investing at age 25\], my view is that there will be tensi…
05. [ 7/10] Trump_s_New_Tariffs_How_

([(7,
   Document(id='100e64f8-9fca-4882-adc5-eaa9b5258c57', metadata={'source': 'Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md', 'thread_title': 'Trump’s New Tariffs – How Are You Adjusting Your Investments?', 'comment_idx': 111}, page_content='Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments?\nComment: (5) NVDA has almost propped up the market in its own. It was bound for a correction, as is the rest of the market. I would be careful making big adjustments but simply reallocating is a great idea. For older investors I’ve moved a portion of FI/Bonds into Value stocks, small/smid cap, and NTMFs. Having alt exposure as an uncorrelated diversifier has been incredibly fruitful for my clients both young and old. I also pulled a portion of any Emerging Markets/Intl exposure to the same above. My thought process is that geopolitical turmoil (not necessarily catastrophic) will be the theme of this presidency. Constant bullying, childish rhetoric, etc. will 